In [18]:
import re
import torch
import stable_worldmodel as swm
from stable_worldmodel.policy import PlanConfig, WorldModelPolicy
from stable_worldmodel.solver import CEMSolver
from stable_worldmodel.wm.utils import _resolve
from stable_worldmodel.data import get_cache_dir
from hydra.utils import instantiate
from random import randint


def _remap_hf_vit_keys(state_dict):
    """Remap old transformers ViT key names (pre-5.x) to the new format."""
    new_sd = {}
    for k, v in state_dict.items():
        k = re.sub(r'^encoder\.encoder\.layer\.(\d+)\.attention\.attention\.query\.', r'encoder.layers.\1.attention.q_proj.', k)
        k = re.sub(r'^encoder\.encoder\.layer\.(\d+)\.attention\.attention\.key\.', r'encoder.layers.\1.attention.k_proj.', k)
        k = re.sub(r'^encoder\.encoder\.layer\.(\d+)\.attention\.attention\.value\.', r'encoder.layers.\1.attention.v_proj.', k)
        k = re.sub(r'^encoder\.encoder\.layer\.(\d+)\.attention\.output\.dense\.', r'encoder.layers.\1.attention.o_proj.', k)
        k = re.sub(r'^encoder\.encoder\.layer\.(\d+)\.intermediate\.dense\.', r'encoder.layers.\1.mlp.fc1.', k)
        k = re.sub(r'^encoder\.encoder\.layer\.(\d+)\.output\.dense\.', r'encoder.layers.\1.mlp.fc2.', k)
        k = re.sub(r'^encoder\.encoder\.layer\.(\d+)\.', r'encoder.layers.\1.', k)
        new_sd[k] = v
    return new_sd


def load_pretrained_compat(name):
    """load_pretrained with HF ViT key remapping for transformers >= 5.x."""
    cache_dir = get_cache_dir(None, sub_folder='checkpoints')
    checkpoint_path, config = _resolve(name, cache_dir)
    state_dict = torch.load(checkpoint_path, map_location='cpu')
    state_dict = _remap_hf_vit_keys(state_dict)
    model = instantiate(config)
    model.load_state_dict(state_dict)
    return model

In [19]:
dataset = swm.data.load_dataset(
    'tworooms.lance',
    num_steps=3,
    frameskip=1,
    keys_to_load=['pixels', 'action', 'state'],
)

In [20]:
device = 'mps'
model = load_pretrained_compat('quentinll/lewm-tworooms').to(device).eval()
model.requires_grad_(False)

num_envs=1

world = swm.World(
    'swm/TwoRoom-v1',
    num_envs=num_envs,
    image_shape=(96, 96),
    max_episode_steps=100,
)

solver = CEMSolver(
    model=model,
    num_samples=300,
    n_steps=10,
    device=device,
)

policy = WorldModelPolicy(
    solver=solver,
    config=PlanConfig(
        horizon=3,
        receding_horizon=3
    ),
)

max_episode = 256
episodes_idx = [10]
#episodes_idx = [randint(0, max_episode-1) for _ in range(num_envs)]
#min_episode_length = min([dataset.load_episode(ep)['action'].shape[0] for ep in episodes_idx])

world.set_policy(policy)
results = world.evaluate(
    dataset=dataset,
    episodes_idx=episodes_idx,
    start_steps=[0 for _ in range(num_envs)],
    goal_offset=10,
    eval_budget=10,
    video='videos/tworooms',
    callables=[
        {'method': '_set_state', 'args': {'state': {'value': 'state'}}},
        {
            'method': '_set_goal_state',
            'args': {'goal_state': {'value': 'goal_state'}},
        },
    ],
)
world.close()

print(results)

16:44:58 | INFO  | utils.py    | Loading quentinll/lewm-tworooms from local cache...
16:44:58 | INFO  | utils.py    | Loading checkpoint from folder /Users/matheoledevehat/.stable_worldmodel/checkpoints/models--quentinll--lewm-tworooms...
16:44:58 | INFO  | utils.py    | Created ViT-tiny from scratch with config: {'hidden_size': 192, 'num_hidden_layers': 12, 'num_attention_heads': 3, 'intermediate_size': 768, 'image_size': 224, 'patch_size': 14}


/Users/matheoledevehat/Code/leworldmodel/.venv/lib/python3.11/site-packages/gymnasium/utils/passive_env_checker.py:130: UserWarning: WARN: The obs returned by the `reset()` method was expecting a numpy array, actual type: <class 'torch.Tensor'>
  logger.warn(
/Users/matheoledevehat/Code/leworldmodel/.venv/lib/python3.11/site-packages/gymnasium/spaces/box.py:424: UserWarning: WARN: Casting input x to numpy array.
  gym.logger.warn("Casting input x to numpy array.")


ValueError: Make sure that the channel dimension of the pixel values match with the one set in the configuration. Expected 3 but got 96.